# 01 - Data Collection: El Clasico All-Time

---

## Objectives
1. Collect El Clasico match data from multiple web sources
2. Work with different data formats (HTML, CSV, JSON)
3. Use BeautifulSoup for static pages and Selenium for dynamic pages

## Data Sources (All Free, No API Keys)

| Source | Method | Data |
|--------|--------|------|
| Wikipedia | pandas.read_html / BeautifulSoup | Complete match list (1902-present) |
| FBref | BeautifulSoup / Selenium | Advanced stats (xG, possession, shots) |
| Transfermarkt | BeautifulSoup | Lineups, market values, match events |
| ESPN FC | Selenium | Match events, scores (dynamic page) |

## Setup

In [ ]:
# Install required packages (run once)
# !pip install requests beautifulsoup4 pandas lxml html5lib selenium webdriver-manager

In [ ]:
import requests
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd
import numpy as np
import json
import os
import time
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Libraries loaded.')

---

## 1. Wikipedia Scraping (BeautifulSoup)

Wikipedia has a dedicated page listing every El Clasico match ever played.  
We use two approaches:
- `pandas.read_html()` for quick table extraction
- `BeautifulSoup` for finer control and link extraction

In [ ]:
WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_El_Cl%C3%A1sico_matches'
print(f'Source: {WIKI_URL}')

### 1.1 Quick extraction with pandas.read_html()

In [ ]:
# pandas.read_html needs a User-Agent header to avoid 403
resp = requests.get(WIKI_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

wiki_tables = pd.read_html(StringIO(resp.text))
print(f'Found {len(wiki_tables)} tables\n')

for i, t in enumerate(wiki_tables):
    print(f'  Table {i}: {t.shape[0]} rows x {t.shape[1]} cols')

In [ ]:
# Pick the main matches table
wiki_matches = None

for i, t in enumerate(wiki_tables):
    cols_lower = [str(c).lower() for c in t.columns]
    if any(kw in c for c in cols_lower for kw in ('date', 'score', 'competition')):
        wiki_matches = t
        print(f'Using Table {i} (matched by column names)')
        break

if wiki_matches is None:
    wiki_matches = max(wiki_tables, key=len)
    print('Using the largest table')

print(f'Shape: {wiki_matches.shape}')
print(f'Columns: {wiki_matches.columns.tolist()}')
wiki_matches.head(10)

### 1.2 Detailed extraction with BeautifulSoup

In [ ]:
resp = requests.get(WIKI_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'lxml')
print(f'Page title: {soup.title.string}')

wikitables = soup.find_all('table', class_='wikitable')
print(f'Found {len(wikitables)} wikitables')

In [ ]:
# Extract rows with links (useful for match-report URLs)
all_rows = []

for table in wikitables:
    header_row = table.find('tr')
    if not header_row:
        continue
    col_names = [th.get_text(strip=True) for th in header_row.find_all(['th', 'td'])]

    for row in table.find_all('tr')[1:]:
        cells = row.find_all(['td', 'th'])
        if not cells:
            continue
        row_data = {}
        for idx, cell in enumerate(cells):
            col = col_names[idx] if idx < len(col_names) else f'col_{idx}'
            row_data[col] = cell.get_text(strip=True)

            link = cell.find('a', href=True)
            if link:
                row_data[f'{col}_link'] = 'https://en.wikipedia.org' + link['href']

        all_rows.append(row_data)

wiki_bs4_df = pd.DataFrame(all_rows)
print(f'Extracted {len(wiki_bs4_df)} rows with BeautifulSoup')
wiki_bs4_df.head()

In [ ]:
# Save both versions
wiki_matches.to_csv(os.path.join(RAW_DIR, 'wikipedia_matches_pandas.csv'), index=False)
wiki_bs4_df.to_csv(os.path.join(RAW_DIR, 'wikipedia_matches_bs4.csv'), index=False)

# Also save as JSON
wiki_matches.to_json(os.path.join(RAW_DIR, 'wikipedia_matches.json'), orient='records', indent=2)

print('Wikipedia data saved (CSV + JSON).')

---

## 2. FBref Scraping (BeautifulSoup)

FBref has advanced match statistics (xG, possession, passing, etc.)  
We scrape their head-to-head page.

Note: FBref rate-limits requests. We add a delay between calls.

In [ ]:
FBREF_H2H_URL = 'https://fbref.com/en/stathead/matchup/teams/53a199f4/206d90db/'
# 53a199f4 = Real Madrid, 206d90db = Barcelona

print(f'Source: {FBREF_H2H_URL}')
print('Note: wait 4 seconds to respect rate limits')

In [ ]:
time.sleep(4)  # rate limiting

try:
    fbref_resp = requests.get(FBREF_H2H_URL, headers=HEADERS, timeout=30)
    fbref_resp.raise_for_status()

    fbref_tables = pd.read_html(StringIO(fbref_resp.text))
    print(f'Found {len(fbref_tables)} tables')

    fbref_matches = max(fbref_tables, key=len)
    print(f'Main table: {fbref_matches.shape}')
    print(f'Columns: {fbref_matches.columns.tolist()}')

    fbref_matches.to_csv(os.path.join(RAW_DIR, 'fbref_head_to_head.csv'), index=False)
    print('Saved FBref data.')
    fbref_matches.head()

except Exception as e:
    print(f'FBref request failed: {e}')
    print('FBref uses Cloudflare protection and blocks automated requests.')
    print('Fallback: use Selenium in Section 4 below.')

---

## 3. Transfermarkt Scraping (BeautifulSoup)

Transfermarkt has match details, lineups, and player market values.

In [ ]:
TM_URL = 'https://www.transfermarkt.com/real-madrid/bilanz/verein/418/gegner_id/131'
# Real Madrid (418) vs Barcelona (131)

print(f'Source: {TM_URL}')

In [ ]:
time.sleep(3)  # rate limiting

tm_headers = {
    'User-Agent': HEADERS['User-Agent'],
    'Accept-Language': 'en-US,en;q=0.9',
}

try:
    tm_resp = requests.get(TM_URL, headers=tm_headers, timeout=30)
    tm_resp.raise_for_status()

    tm_soup = BeautifulSoup(tm_resp.text, 'lxml')
    tm_tables = pd.read_html(StringIO(str(tm_soup)), flavor='lxml')

    print(f'Found {len(tm_tables)} tables')
    for i, t in enumerate(tm_tables):
        print(f'  Table {i}: {t.shape}')

    if tm_tables:
        tm_main = max(tm_tables, key=len)
        tm_main.to_csv(os.path.join(RAW_DIR, 'transfermarkt_h2h.csv'), index=False)
        print('Saved Transfermarkt data.')
        tm_main.head()

except Exception as e:
    print(f'Transfermarkt request failed: {e}')
    print('Transfermarkt blocks automated requests (HTTP 405/403).')
    print('Try using Selenium or a VPN.')

---

## 4. Dynamic Scraping with Selenium

Some websites render content using JavaScript. Selenium automates a real
browser to load these pages.

Requirements:
- Chrome browser installed
- `pip install selenium webdriver-manager`

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

print('Selenium imported.')

In [ ]:
def scrape_with_selenium(url, wait_seconds=6):
    """Load a JavaScript-rendered page and extract all tables."""

    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument(
        '--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    try:
        print(f'Loading: {url}')
        driver.get(url)
        time.sleep(wait_seconds)

        html = driver.page_source
        tables = pd.read_html(StringIO(html))
        print(f'Found {len(tables)} tables')
        return tables
    finally:
        driver.quit()
        print('WebDriver closed.')

In [ ]:
# Example: FBref via Selenium (uncomment to run)

# fbref_selenium_tables = scrape_with_selenium(FBREF_H2H_URL)
# if fbref_selenium_tables:
#     fbref_sel = max(fbref_selenium_tables, key=len)
#     fbref_sel.to_csv(os.path.join(RAW_DIR, 'fbref_selenium.csv'), index=False)
#     print(f'Saved {len(fbref_sel)} rows')
#     fbref_sel.head()

print('Selenium example ready. Uncomment the cell above to run.')

In [ ]:
# Example: ESPN FC via Selenium (uncomment to run)

# espn_url = 'https://www.espn.com/soccer/team/fixtures/_/id/86/real-madrid'
# espn_tables = scrape_with_selenium(espn_url, wait_seconds=8)
# if espn_tables:
#     for i, t in enumerate(espn_tables):
#         print(f'Table {i}: {t.shape}')

print('ESPN example ready. Uncomment the cell above to run.')

---

## 5. Data Summary

In [ ]:
print('Collected Data Files:')
print('=' * 50)

for f in sorted(os.listdir(RAW_DIR)):
    path = os.path.join(RAW_DIR, f)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {f:45s} {size_kb:8.1f} KB')

In [ ]:
# Quick look at the Wikipedia dataset
df = pd.read_csv(os.path.join(RAW_DIR, 'wikipedia_matches_pandas.csv'))
print(f'Wikipedia dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nColumn list: {df.columns.tolist()}')
print(f'\nData types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')

---

## Next Steps

Continue to `02_data_cleaning.ipynb` for:

1. Deduplication
2. Missing value imputation: Simple, KNN, MICE
3. Normalization and standardization
4. Outlier detection: Z-Score, IQR, K-Means
5. Data merging and reshaping